In [1]:
import fire

import qlib
import pickle
from qlib.constant import REG_CN
from qlib.config import HIGH_FREQ_CONFIG

from qlib.utils import init_instance_by_config
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.ops import Operators
from qlib.data.data import Cal
from qlib.tests.data import GetData

from highfreq_ops import get_calendar_day, DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut

In [3]:
# ============================================
# KQDownloader_fixed.py 调用区块（Qlib兼容）
# ============================================
from pathlib import Path
import os
import sys
import subprocess
from datetime import datetime

# 自动识别路径（支持从 qlib/examples/highfreq/ 中运行）
nb_cwd = Path.cwd()
if (nb_cwd / 'scripts').exists() and (nb_cwd / 'qlib').exists():
    PROJECT_ROOT = nb_cwd
else:
    PROJECT_ROOT = nb_cwd
    if PROJECT_ROOT.name.lower() == 'highfreq' and PROJECT_ROOT.parent.name.lower() == 'examples':
        PROJECT_ROOT = PROJECT_ROOT.parents[1]

# === 参数配置 ===
SCRIPT = PROJECT_ROOT / "scripts" / "data_collector" / "KQ" / "KQdownloader.py"
RAW_DIR = PROJECT_ROOT.parent / "kq_raw_data"       # 临时保存天勤原始CSV
QLIB_DIR = PROJECT_ROOT.parent / "qlib_data"        # 输出为Qlib标准数据
POOL_CSV = Path(r"C:\Users\ASUS\qlib\examples\highfreq\sorted_high_preclose_ratio_2025.csv")

START = "2025-01-01"
END = "2025-09-30"
INTERVAL = "1min"          # Qlib分钟级数据
USERNAME = "xclight"
PASSWORD = "xclight666"
LIMIT_NUMS = None         # 下载股票数量限制（None 表示全部）



# === benchmark 配置 ===
BENCHMARK_CODE = "SSE.000300"  # 沪深300
BENCHMARK_DIR = QLIB_DIR / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)  # 确保目录存在
BENCHMARK_FILE = BENCHMARK_DIR / f"{BENCHMARK_CODE}.csv"

# === 路径检查 ===
RAW_DIR.mkdir(parents=True, exist_ok=True)
QLIB_DIR.mkdir(parents=True, exist_ok=True)

print("📁 当前工作路径:", nb_cwd)
print("🧭 项目根目录:", PROJECT_ROOT)
print("📜 使用脚本:", SCRIPT)
print("📦 原始数据目录:", RAW_DIR)
print("📊 Qlib数据输出目录:", QLIB_DIR)
print("📋 股票池:", POOL_CSV)
print("📊 Benchmark:", BENCHMARK_CODE)


# === 构建命令 ===
cmd = [
    sys.executable, str(SCRIPT), 'run',
    '--source_dir', str(RAW_DIR),
    '--target_dir', str(QLIB_DIR),
    '--csv_stock_pool', str(POOL_CSV),
    '--start', START,
    '--end', END,
    '--interval', INTERVAL,
    '--username', USERNAME,
    '--password', PASSWORD,
    '--benchmark', BENCHMARK_CODE,  # ⚙️ 新增 benchmark 参数
    '--benchmark_dir', str(BENCHMARK_DIR)  # ⚙️ benchmark 存放路径
]
if LIMIT_NUMS:
    cmd += ['--limit_nums', str(LIMIT_NUMS)]

print("\n🚀 启动下载命令:\n", ' '.join(map(str, cmd)), "\n")

# === 启动下载过程（实时输出日志） ===
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"❌ Downloader failed with return code {rc}")

print("\n✅ 数据下载与 Qlib 格式转换已完成！")
print(f"📂 Qlib 数据位置: {QLIB_DIR}")
print(f"📊 Benchmark 数据位置: {BENCHMARK_FILE}")


📁 当前工作路径: c:\Users\ASUS\qlib\examples\highfreq
🧭 项目根目录: c:\Users\ASUS\qlib
📜 使用脚本: c:\Users\ASUS\qlib\scripts\data_collector\KQ\KQdownloader.py
📦 原始数据目录: c:\Users\ASUS\kq_raw_data
📊 Qlib数据输出目录: c:\Users\ASUS\qlib_data
📋 股票池: C:\Users\ASUS\qlib\examples\highfreq\sorted_high_preclose_ratio_2025.csv
📊 Benchmark: SSE.000300

🚀 启动下载命令:
 c:\Users\ASUS\miniconda3\python.exe c:\Users\ASUS\qlib\scripts\data_collector\KQ\KQdownloader.py run --source_dir c:\Users\ASUS\kq_raw_data --target_dir c:\Users\ASUS\qlib_data --csv_stock_pool C:\Users\ASUS\qlib\examples\highfreq\sorted_high_preclose_ratio_2025.csv --start 2025-01-01 --end 2025-09-30 --interval 1min --username xclight --password xclight666 --benchmark SSE.000300 --benchmark_dir c:\Users\ASUS\qlib_data\benchmark 

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/
2025-10-11 13:46:52.260 | INFO     | __main__:run:125 - ⚙️ 未设置 limit_nums，默认下载全部股票
2025-10-11 13:46:52.260 | INFO     | __main__:run:127 - 📊 股票数: 5

In [8]:
#下载数据预处理 转化成qlib数据格式完全一样
import os
import pandas as pd
import numpy as np
from pathlib import Path
import struct

# ====== 天勤数据目录（直接覆盖） ======
KQ_DATA_DIR = Path(r"C:\Users\ASUS\qlib_data")  # 天勤下载后的根目录

# ====== 工具函数 ======
def market_prefix_convert(code: str):
    """sse.xxx / szse.xxx → SHxxx / SZxxx (文件夹名用大写)"""
    code = code.upper()
    if code.startswith("SSE."):
        return "SH" + code.split(".")[1]
    elif code.startswith("SZSE."):
        return "SZ" + code.split(".")[1]
    else:
        return code.upper()

def market_prefix_convert_upper(code: str):
    """sse.xxx / szse.xxx → SHxxx / SZxxx (all.txt用大写)"""
    code = code.upper()
    if code.startswith("SSE."):
        return "SH" + code.split(".")[1]
    elif code.startswith("SZSE."):
        return "SZ" + code.split(".")[1]
    else:
        return code.upper()

# 测试转换函数
print("🧪 测试转换函数:")
test_codes = ["sse.000300", "szse.000001", "SSE.600519", "SZSE.000858"]
for code in test_codes:
    converted_folder = market_prefix_convert(code)
    converted_file = market_prefix_convert_upper(code)
    print(f"   {code} → 文件夹: {converted_folder}, all.txt: {converted_file}")

def write_bin(series: pd.Series, path: Path):
    """写入 float32 二进制"""
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = series.astype(np.float32).to_numpy()
    with open(path, "wb") as f:
        f.write(struct.pack(f"{len(arr)}f", *arr))

def read_bin(fp):
    return np.fromfile(fp, dtype=np.float32)

# ====== 1. 日历文件保持不变 ======
print(f"✅ 日历文件已存在: {KQ_DATA_DIR / 'calendars' / '1min.txt'}")

# ====== 2. instruments/all.txt 转换 ======
inst_src = KQ_DATA_DIR / "instruments" / "all.txt"

# 使用更灵活的读取方式处理格式问题
print(f"🔍 检查股票池文件格式: {inst_src}")

# 先读取原始内容查看格式
with open(inst_src, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    
print(f"   文件总行数: {len(lines)}")
print(f"   前3行原始内容:")
for i in range(min(3, len(lines))):
    print(f"     {repr(lines[i].strip())}")

# 手动解析每行，处理引号和格式问题
parsed_data = []
for line in lines:
    line = line.strip()
    if not line:
        continue
    
    # 分割字段，处理引号问题
    parts = line.split('\t')
    if len(parts) >= 3:
        instrument = parts[0].strip()
        start_time = parts[1].strip()
        end_time = parts[2].strip().strip('"').strip()  # 去除引号和空格
        
        # 清理end_time中的多余内容
        if '"' in end_time:
            end_time = end_time.split('"')[0].strip()
        
        parsed_data.append([instrument, start_time, end_time])
    else:
        print(f"⚠️ 跳过格式异常的行: {line}")

# 创建DataFrame
inst_df = pd.DataFrame(parsed_data, columns=["instrument", "start", "end"])

print(f"📝 解析完成，数据形状: {inst_df.shape}")
print(f"   前5行解析结果:")
for i in range(min(5, len(inst_df))):
    print(f"     {inst_df.iloc[i].tolist()}")

# 转换股票代码格式（all.txt用大写）
inst_df["instrument"] = inst_df["instrument"].apply(market_prefix_convert_upper)

# 保存转换后的文件
inst_df.to_csv(inst_src, sep="\t", header=False, index=False)
print(f"✅ 股票池格式已转换: {inst_src}")
print(f"   转换后形状: {inst_df.shape}")
print(f"   前5支股票: {inst_df['instrument'].head().tolist()}")

# ====== 3. features 特征字段补齐 ======
feature_src = KQ_DATA_DIR / "features"

# 删除benchmark文件夹（如果存在）
benchmark_dir = KQ_DATA_DIR / "benchmark"
if benchmark_dir.exists():
    import shutil
    shutil.rmtree(benchmark_dir)
    print(f"🗑️ 已删除benchmark文件夹: {benchmark_dir}")

print(f"📁 开始处理特征目录: {feature_src}")

# 处理每个股票的特征文件（先处理，后重命名）
print(f"📁 开始处理特征文件...")

for inst_dir in feature_src.iterdir():
    if not inst_dir.is_dir():
        continue

    files = {f.stem.split(".")[0]: f for f in inst_dir.glob("*.1min.bin")}
    print(f"📈 处理 {inst_dir.name}: {list(files.keys())}")

    # 加载基础字段
    open_ = read_bin(files.get("open"))
    high = read_bin(files.get("high"))
    low = read_bin(files.get("low"))
    close = read_bin(files.get("close"))
    vol = read_bin(files.get("volume"))

    # 构造补充字段
    factor = np.ones_like(close, dtype=np.float32)
    paused = np.zeros_like(close, dtype=np.float32)
    paused_num = np.zeros_like(close, dtype=np.float32)
    change = np.concatenate([[0], np.diff(close)])  # ✅ 与 Qlib 一致：涨跌额

    # 写入
    write_bin(pd.Series(open_), inst_dir / "open.1min.bin")
    write_bin(pd.Series(high), inst_dir / "high.1min.bin")
    write_bin(pd.Series(low), inst_dir / "low.1min.bin")
    write_bin(pd.Series(close), inst_dir / "close.1min.bin")
    write_bin(pd.Series(vol), inst_dir / "volume.1min.bin")
    write_bin(pd.Series(factor), inst_dir / "factor.1min.bin")
    write_bin(pd.Series(paused), inst_dir / "paused.1min.bin")
    write_bin(pd.Series(paused_num), inst_dir / "paused_num.1min.bin")
    write_bin(pd.Series(change), inst_dir / "change.1min.bin")

print(f"✅ 特征文件处理完成")

# 现在重命名所有文件夹为大写格式
print(f"\n🔄 开始重命名文件夹为大写格式...")

# 收集所有需要重命名的目录
rename_operations = []
print(f"🔍 检查所有目录:")
for inst_dir in feature_src.iterdir():
    if not inst_dir.is_dir():
        continue
    
    # 转换文件夹名称为Qlib标准格式（大写）
    inst_code = market_prefix_convert(inst_dir.name)
    target_dir = feature_src / inst_code
    
    print(f"   目录: {inst_dir.name} → 转换后: {inst_code}")
    
    if inst_dir != target_dir:
        rename_operations.append((inst_dir, target_dir))
        print(f"   ✅ 需要重命名: {inst_dir.name} → {inst_code}")
    else:
        print(f"   ⏭️ 无需重命名: {inst_dir.name}")

print(f"🔄 需要重命名的目录数量: {len(rename_operations)}")

# 显示重命名计划
if rename_operations:
    print(f"📋 重命名计划:")
    for i, (inst_dir, target_dir) in enumerate(rename_operations[:10]):  # 只显示前10个
        print(f"   {i+1}. {inst_dir.name} → {target_dir.name}")
    if len(rename_operations) > 10:
        print(f"   ... 还有 {len(rename_operations) - 10} 个目录")

# 执行重命名操作
for inst_dir, target_dir in rename_operations:
    try:
        print(f"📝 重命名: {inst_dir.name} → {target_dir.name}")
        os.rename(inst_dir, target_dir)
    except Exception as e:
        print(f"❌ 重命名失败 {inst_dir.name}: {e}")

print(f"✅ 目录重命名完成")

print("🎉 天勤分钟数据预处理完成，已覆盖为 Qlib 兼容格式！")

# ====== 4. 验证数据转换结果 ======
print("\n🔍 开始验证数据转换结果...")

# 验证日历文件
calendar_file = KQ_DATA_DIR / "calendars" / "1min.txt"
if calendar_file.exists():
    with open(calendar_file, 'r') as f:
        calendar_lines = f.readlines()
    print(f"✅ 日历文件验证: {len(calendar_lines)} 个交易日")
    print(f"   最早交易日: {calendar_lines[0].strip()}")
    print(f"   最晚交易日: {calendar_lines[-1].strip()}")
else:
    print("❌ 日历文件不存在")

# 验证股票池文件
instruments_file = KQ_DATA_DIR / "instruments" / "all.txt"
if instruments_file.exists():
    inst_df_check = pd.read_csv(instruments_file, header=None, sep='\t')
    print(f"✅ 股票池文件验证: {len(inst_df_check)} 支股票")
    print(f"   前5支股票: {inst_df_check.iloc[:5, 0].tolist()}")
else:
    print("❌ 股票池文件不存在")

# 验证特征文件
feature_dir = KQ_DATA_DIR / "features"
if feature_dir.exists():
    stock_dirs = [d for d in feature_dir.iterdir() if d.is_dir()]
    print(f"✅ 特征目录验证: {len(stock_dirs)} 个股票目录")
    
    # 检查第一个股票的特征文件
    if stock_dirs:
        first_stock = stock_dirs[0]
        feature_files = list(first_stock.glob("*.1min.bin"))
        print(f"   示例股票 {first_stock.name} 的特征文件:")
        for f in feature_files:
            file_size = f.stat().st_size
            data_points = file_size // 4  # float32 = 4 bytes
            print(f"     {f.name}: {data_points} 个数据点")
            
        # 验证数据内容
        try:
            close_data = read_bin(first_stock / "close.1min.bin")
            print(f"    {first_stock.name} close 数据范围: {close_data.min():.2f} ~ {close_data.max():.2f}")
            print(f"    数据点数: {len(close_data)}")
        except Exception as e:
            print(f"   ❌ 读取数据失败: {e}")
else:
    print("❌ 特征目录不存在")

# 验证benchmark文件夹（已删除）
benchmark_dir = KQ_DATA_DIR / "benchmark"
if benchmark_dir.exists():
    print("⚠️ Benchmark文件夹仍然存在")
else:
    print("✅ Benchmark文件夹已删除（符合Qlib标准）")

# 最终验证总结
print(f"\n📊 数据转换验证总结:")
print(f"   数据目录: {KQ_DATA_DIR}")
print(f"   日历文件: {'✅' if calendar_file.exists() else '❌'}")
print(f"   股票池文件: {'✅' if instruments_file.exists() else '❌'}")
print(f"   特征目录: {'✅' if feature_dir.exists() else '❌'}")
print(f"   Benchmark文件夹: {'❌' if benchmark_dir.exists() else '✅'} (已删除)")

# 检查文件夹命名格式
print(f"\n🔍 检查文件夹命名格式:")
if feature_dir.exists():
    stock_dirs = [d for d in feature_dir.iterdir() if d.is_dir()]
    if stock_dirs:
        print(f"   前5个文件夹名称:")
        for i, d in enumerate(stock_dirs[:5]):
            print(f"     {i+1}. {d.name}")
        
        # 检查是否符合Qlib标准格式（文件夹用大写）
        non_standard = [d.name for d in stock_dirs if not (d.name.startswith('SH') or d.name.startswith('SZ'))]
        if non_standard:
            print(f"   ⚠️ 发现非标准格式文件夹: {non_standard[:5]}")
        else:
            print(f"   ✅ 所有文件夹都符合Qlib标准格式 (SH/SZ开头，大写)")
    else:
        print("   ❌ 没有找到股票文件夹")

# 检查是否有任何文件被修改
import time
current_time = time.time()
recent_files = []

for root, dirs, files in os.walk(KQ_DATA_DIR):
    for file in files:
        file_path = Path(root) / file
        if file_path.stat().st_mtime > current_time - 300:  # 5分钟内的文件
            recent_files.append(str(file_path))

if recent_files:
    print(f"\n🔄 最近5分钟内修改的文件 ({len(recent_files)} 个):")
    for f in recent_files[:10]:  # 只显示前10个
        print(f"   {f}")
    if len(recent_files) > 10:
        print(f"   ... 还有 {len(recent_files) - 10} 个文件")
else:
    print(f"\n⚠️ 警告: 最近5分钟内没有文件被修改，可能数据转换未成功执行")

print("\n🎯 验证完成！如果所有项目都显示 ✅，说明数据转换成功。")



🧪 测试转换函数:
   sse.000300 → 文件夹: SH000300, all.txt: SH000300
   szse.000001 → 文件夹: SZ000001, all.txt: SZ000001
   SSE.600519 → 文件夹: SH600519, all.txt: SH600519
   SZSE.000858 → 文件夹: SZ000858, all.txt: SZ000858
✅ 日历文件已存在: C:\Users\ASUS\qlib_data\calendars\1min.txt
🔍 检查股票池文件格式: C:\Users\ASUS\qlib_data\instruments\all.txt
   文件总行数: 501
   前3行原始内容:
     'SH000300\t2025-01-02 09:30:00\t2025-09-29 14:59:00'
     'SH600105\t2025-01-02 09:30:00\t2025-09-29 14:59:00'
     'SH600126\t2025-01-02 09:30:00\t2025-09-29 14:59:00'
📝 解析完成，数据形状: (501, 3)
   前5行解析结果:
     ['SH000300', '2025-01-02 09:30:00', '2025-09-29 14:59:00']
     ['SH600105', '2025-01-02 09:30:00', '2025-09-29 14:59:00']
     ['SH600126', '2025-01-02 09:30:00', '2025-09-29 14:59:00']
     ['SH600246', '2025-01-02 09:30:00', '2025-09-29 14:59:00']
     ['SH600353', '2025-01-02 09:30:00', '2025-09-29 14:59:00']
✅ 股票池格式已转换: C:\Users\ASUS\qlib_data\instruments\all.txt
   转换后形状: (501, 3)
   前5支股票: ['SH000300', 'SH600105', 'SH600126', 'SH60

In [11]:


from pathlib import Path

FEATURES_DIR = Path(r"C:\Users\ASUS\qlib_data\features")

for folder in FEATURES_DIR.iterdir():
    if folder.is_dir():
        temp_name = folder.name + "_tmp"
        temp_path = folder.parent / temp_name
        folder.rename(temp_path)  # 先改临时名字
        final_name = folder.name.upper()
        final_path = folder.parent / final_name
        temp_path.rename(final_path)  # 再改成大写
        print(f"✅ 重命名: {folder.name} -> {final_name}")

print("🎉 features 文件夹全部改为大写完成！")



✅ 重命名: sh000300 -> SH000300
✅ 重命名: sh600105 -> SH600105
✅ 重命名: sh600126 -> SH600126
✅ 重命名: sh600246 -> SH600246
✅ 重命名: sh600353 -> SH600353
✅ 重命名: sh600397 -> SH600397
✅ 重命名: sh600410 -> SH600410
✅ 重命名: sh600530 -> SH600530
✅ 重命名: sh600580 -> SH600580
✅ 重命名: sh600589 -> SH600589
✅ 重命名: sh600590 -> SH600590
✅ 重命名: sh600592 -> SH600592
✅ 重命名: sh600610 -> SH600610
✅ 重命名: sh600698 -> SH600698
✅ 重命名: sh600735 -> SH600735
✅ 重命名: sh600967 -> SH600967
✅ 重命名: sh601086 -> SH601086
✅ 重命名: sh601606 -> SH601606
✅ 重命名: sh601869 -> SH601869
✅ 重命名: sh603011 -> SH603011
✅ 重命名: sh603040 -> SH603040
✅ 重命名: sh603063 -> SH603063
✅ 重命名: sh603072 -> SH603072
✅ 重命名: sh603083 -> SH603083
✅ 重命名: sh603086 -> SH603086
✅ 重命名: sh603090 -> SH603090
✅ 重命名: sh603119 -> SH603119
✅ 重命名: sh603124 -> SH603124
✅ 重命名: sh603127 -> SH603127
✅ 重命名: sh603130 -> SH603130
✅ 重命名: sh603166 -> SH603166
✅ 重命名: sh603171 -> SH603171
✅ 重命名: sh603200 -> SH603200
✅ 重命名: sh603226 -> SH603226
✅ 重命名: sh603228 -> SH603228
✅ 重命名: sh603256 -> S

筛选股票 有的股票起始时间不从01-02开始

In [4]:
from pathlib import Path

# 检查股票数据完整性，筛选有完整数据的股票
print("🔍 检查股票数据完整性，筛选有完整数据的股票:")
print("="*60)

# 读取 instruments/all.txt
instruments_file = Path("C:/Users/ASUS/qlib_data/instruments/all.txt")
if instruments_file.exists():
    with open(instruments_file, 'r') as f:
        lines = f.readlines()
    
    print(f"总股票数: {len(lines)}")
    
    # 检查每个股票的数据时间范围
    complete_stocks = []
    incomplete_stocks = []
    
    for i, line in enumerate(lines):
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            code = parts[0].strip()
            start_time = parts[1].strip()
            end_time = parts[2].strip()
            
            # 检查是否覆盖所需时间段
            required_start = "2025-01-02 09:30:00"
            required_end = "2025-09-29 14:59:00"
            
            # 检查开始时间
            start_ok = start_time <= required_start
            # 检查结束时间  
            end_ok = required_end <= end_time
            
            if start_ok and end_ok:
                complete_stocks.append(code)
            else:
                incomplete_stocks.append((code, start_time, end_time))
    
    print(f"\n完整数据股票数: {len(complete_stocks)}")
    print(f"不完整数据股票数: {len(incomplete_stocks)}")

    # 显示不完整的股票
    if incomplete_stocks:
        print("\n不完整数据股票示例:")
        for code, start, end in incomplete_stocks[:10]:
            print(f"  {code}: {start} ~ {end}")
    
    # 显示完整的股票
    if complete_stocks:
        print(f"\n完整数据股票示例:")
        for code in complete_stocks[:10]:
            print(f"  {code}")
    
    # 生成筛选后的股票池
    print(f"\n生成筛选后的股票池:")
    filtered_file = Path("C:/Users/ASUS/qlib_data/instruments/all_filtered.txt")
    with open(filtered_file, 'w') as f:
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                code = parts[0].strip()
                if code in complete_stocks:
                    f.write(line)
    
    print(f"筛选后股票数: {len(complete_stocks)}")
    print(f"筛选后文件: {filtered_file}")
    
    # 建议修改配置
    print(f"\n建议修改配置:")
    print(f"使用筛选后的股票池: {filtered_file}")
    print(f"或者修改 Handler 配置，只使用完整数据的股票")

    filtered_stock_codes = complete_stocks
    print(filtered_stock_codes)

🔍 检查股票数据完整性，筛选有完整数据的股票:
总股票数: 501

完整数据股票数: 471
不完整数据股票数: 30

不完整数据股票示例:
  SH603072: 2025-01-03 09:30:00 ~ 2025-09-29 14:59:00
  SH603124: 2025-03-20 09:30:00 ~ 2025-09-29 14:59:00
  SH603257: 2025-04-08 09:30:00 ~ 2025-09-29 14:59:00
  SH603382: 2025-06-12 09:30:00 ~ 2025-09-29 14:59:00
  SH688411: 2025-01-27 09:30:00 ~ 2025-09-29 14:59:00
  SH688545: 2025-01-22 09:30:00 ~ 2025-09-29 14:59:00
  SH688583: 2025-01-15 09:30:00 ~ 2025-09-29 14:59:00
  SH688757: 2025-03-25 09:30:00 ~ 2025-09-29 14:59:00
  SH688758: 2025-01-10 09:30:00 ~ 2025-09-29 14:59:00
  SH688775: 2025-06-11 09:30:00 ~ 2025-09-29 14:59:00

完整数据股票示例:
  SH000300
  SH600105
  SH600126
  SH600246
  SH600353
  SH600397
  SH600410
  SH600530
  SH600580
  SH600589

生成筛选后的股票池:
筛选后股票数: 471
筛选后文件: C:\Users\ASUS\qlib_data\instruments\all_filtered.txt

建议修改配置:
使用筛选后的股票池: C:\Users\ASUS\qlib_data\instruments\all_filtered.txt
或者修改 Handler 配置，只使用完整数据的股票
['SH000300', 'SH600105', 'SH600126', 'SH600246', 'SH600353', 'SH600397', 'SH600410

数据地址 时间段配置

In [3]:
import os
import sys
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.workflow import R
from qlib.utils import flatten_dict, init_instance_by_config
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.contrib.strategy.signal_strategy import TopkDropoutStrategy
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.model.gbdt import LGBModel
from qlib.backtest import backtest
from workflow import HighfreqWorkflow

# 确保当前路径包含 workflow.py
WORKDIR = r"C:\Users\ASUS\qlib\examples\highfreq"
sys.path.append(WORKDIR)

# 导入自定义操作符
from highfreq_ops import get_calendar_day, DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut

# 配置自定义操作符
SPEC_CONF = {"custom_ops": [DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut], "expression_cache": None}

print("✅ 自定义操作符已配置")

# 天勤数据配置
KQ_DATA_DIR = r"C:\Users\ASUS\qlib_data"
STOCK_POOL_CSV = r"C:\Users\ASUS\qlib\examples\highfreq\sorted_high_preclose_ratio_2025.csv"

# 天勤数据时间段配置
START_TIME = "2025-01-02 09:30:00"
END_TIME = "2025-09-29 14:59:00"
TRAIN_END_TIME = "2025-04-30 14:59:00"  # 1-4月用于训练
TEST_START_TIME = "2025-05-02 09:30:00"  # 5-9月用于测试和回测

print(f"🔧 天勤数据配置:")
print(f"📂 数据目录: {KQ_DATA_DIR}")
print(f"📋 股票池: {STOCK_POOL_CSV}")
print(f"📅 训练时间段: {START_TIME} ~ {TRAIN_END_TIME}")
print(f"📅 测试时间段: {TEST_START_TIME} ~ {END_TIME}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError. CatBoostModel are skipped. (optional: maybe installing CatBoostModel can fix it.)
ModuleNotFoundError. XGBModel is skipped(optional: maybe installing xgboost can fix it).
ModuleNotFoundError.  PyTorch models are skipped (optional: maybe installing pytorch can fix it).
✅ 自定义操作符已配置
🔧 天勤数据配置:
📂 数据目录: C:\Users\ASUS\qlib_data
📋 股票池: C:\Users\ASUS\qlib\examples\highfreq\sorted_high_preclose_ratio_2025.csv
📅 训练时间段: 2025-01-02 09:30:00 ~ 2025-04-30 14:59:00
📅 测试时间段: 2025-05-02 09:30:00 ~ 2025-09-29 14:59:00


初始化qlib（数据路径自定义 SPEC CONF自定义操作符配置） 筛选有效股票

In [4]:
# 强制使用自定义路径，避免 HIGH_FREQ_CONFIG 的默认路径覆盖
print("🔧 强制设置自定义数据路径...")

# 方法1：直接设置配置，不使用 HIGH_FREQ_CONFIG
CUSTOM_QLIB_CONFIG = {
    "provider_uri": KQ_DATA_DIR,  # 强制使用天勤数据路径
    "dataset_cache": None,
    "expression_cache": "DiskExpressionCache",
    "region": REG_CN,
    **SPEC_CONF  # 添加自定义操作符配置
}

print(f"📂 强制使用数据路径: {KQ_DATA_DIR}")
print(f"🔧 配置内容: {CUSTOM_QLIB_CONFIG}")

# 尝试初始化，如果失败则使用备用方案
try:
    qlib.init(**CUSTOM_QLIB_CONFIG)
    print(f"✅ Qlib 已初始化，使用天勤数据: {KQ_DATA_DIR}")
except Exception as e:
    print(f"⚠️ 自定义路径初始化失败: {e}")
    print("🔄 尝试备用方案：使用默认配置但指定数据路径...")
    
    # 备用方案：使用 HIGH_FREQ_CONFIG 但强制覆盖路径
    from qlib.config import HIGH_FREQ_CONFIG
    BACKUP_CONFIG = HIGH_FREQ_CONFIG.copy()
    BACKUP_CONFIG["provider_uri"] = KQ_DATA_DIR
    BACKUP_CONFIG.update(SPEC_CONF)
    
    qlib.init(**BACKUP_CONFIG)
    print(f"✅ 使用备用配置初始化成功")

print(f"✅ 自定义操作符已加载")

# 验证数据路径是否正确生效
from qlib.config import C
print(f"🔍 验证：当前 Qlib 数据路径: {C.provider_uri}")

# 如果路径仍然不对，提供手动检查
if str(C.provider_uri) != KQ_DATA_DIR:
    print(f"⚠️ 警告：数据路径不匹配！")
    print(f"  期望路径: {KQ_DATA_DIR}")
    print(f"  实际路径: {C.provider_uri}")
    print("💡 建议检查天勤数据转换是否完整")


stock_pool_df = pd.read_csv(STOCK_POOL_CSV)
stock_codes = stock_pool_df["code"].tolist()
print(f"📊 股票池包含 {len(stock_codes)} 支股票")

from qlib.data import D
from pathlib import Path
import pickle

# 读取 instruments 文件
instruments_file = Path("C:/Users/ASUS/qlib_data/instruments/all.txt")
if not instruments_file.exists():
    raise FileNotFoundError(instruments_file)

lines = open(instruments_file).read().splitlines()
codes = [line.split("\t")[0].strip().upper() for line in lines if line.strip()]

start_time = "2025-01-02 09:30:00"
end_time = "2025-09-29 14:59:00"
fields = ["$close"]

print(f"共 {len(codes)} 支股票，开始动态检查是否有数据...")
ok_codes, bad_codes = [], []

for code in codes:
    try:
        df = D.features([code], fields, start_time=start_time, end_time=end_time, freq="1min")
        if df.empty:
            bad_codes.append(code)
        else:
            ok_codes.append(code)
    except Exception:
        bad_codes.append(code)

print(f"\n✅ 有数据: {len(ok_codes)} 支, ❌ 无数据: {len(bad_codes)} 支")

# 保存结果
with open("filtered_market.pkl", "wb") as f:
    pickle.dump(ok_codes, f)
print("✅ 筛选结果已保存 -> filtered_market.pkl")

import pickle

# 读取筛选后的股票列表
with open("filtered_market.pkl", "rb") as f:
    stock_codes = pickle.load(f)

print(f"筛选后股票数量: {len(stock_codes)}")
print(stock_codes[:10])



🔧 强制设置自定义数据路径...
📂 强制使用数据路径: C:\Users\ASUS\qlib_data
🔧 配置内容: {'provider_uri': 'C:\\Users\\ASUS\\qlib_data', 'dataset_cache': None, 'expression_cache': None, 'region': 'cn', 'custom_ops': [<class 'highfreq_ops.DayLast'>, <class 'highfreq_ops.FFillNan'>, <class 'highfreq_ops.BFillNan'>, <class 'highfreq_ops.Date'>, <class 'highfreq_ops.Select'>, <class 'highfreq_ops.IsNull'>, <class 'highfreq_ops.Cut'>]}


[28300:MainThread](2025-10-13 10:48:46,010) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[28300:MainThread](2025-10-13 10:48:46,016) INFO - qlib.Initialization - [__init__.py:79] - qlib successfully initialized based on client settings.
[28300:MainThread](2025-10-13 10:48:46,018) INFO - qlib.Initialization - [__init__.py:81] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/ASUS/qlib_data')}


✅ Qlib 已初始化，使用天勤数据: C:\Users\ASUS\qlib_data
✅ 自定义操作符已加载
🔍 验证：当前 Qlib 数据路径: {'__DEFAULT_FREQ': 'C:\\Users\\ASUS\\qlib_data'}
⚠️ 警告：数据路径不匹配！
  期望路径: C:\Users\ASUS\qlib_data
  实际路径: {'__DEFAULT_FREQ': 'C:\\Users\\ASUS\\qlib_data'}
💡 建议检查天勤数据转换是否完整
📊 股票池包含 500 支股票
共 501 支股票，开始动态检查是否有数据...

✅ 有数据: 501 支, ❌ 无数据: 0 支
✅ 筛选结果已保存 -> filtered_market.pkl
筛选后股票数量: 501
['SH000300', 'SH600105', 'SH600126', 'SH600246', 'SH600353', 'SH600397', 'SH600410', 'SH600530', 'SH600580', 'SH600589']


初始化highfreqworkflow并设置时间

In [5]:
#直接使用workflow.py中的函数定义

wf = HighfreqWorkflow()

wf.start_time = START_TIME
wf.train_end_time = TRAIN_END_TIME
wf.test_start_time = TEST_START_TIME
wf.end_time = END_TIME



构建 handler 配置（指定股票池 + provider数据路径）

In [6]:
# 拷贝 handler 配置并指定 provider_uri
handler_config = wf.DATA_HANDLER_CONFIG0.copy()
handler_config["instruments"] = stock_codes
#handler_config["provider_uri"] = KQ_DATA_DIR   # ⚠️ 关键
handler_config["infer_processors"] = [
    {"class": "HighFreqNorm", "module_path": "highfreq_processor",
     "kwargs": {"fit_start_time": wf.start_time,
                "fit_end_time": wf.train_end_time}}
]



构建 dataset_task 和 model_task

In [13]:
#训练
from qlib.workflow import R
from qlib.utils import init_instance_by_config
from qlib.contrib.model.gbdt import LGBModel
import pandas as pd

# 明确指定训练特征和标签
features_columns = ["FEATURE_%d" % i for i in range(12 * 240)]  # HighFreqNorm 输出的列
#labels_columns = ["Ref($close, -1) / $close - 1"]  # 次日收益率作为示例
labels_columns = ["LABEL0"]
# dataset task
dataset_task = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "HighFreqHandler",
            "module_path": "highfreq_handler",
            "kwargs": handler_config,
        },
        "segments": {
            "train": (wf.start_time, wf.train_end_time),
            "test": (wf.test_start_time, wf.end_time),
        },
        "infer_processors": handler_config["infer_processors"],
        "learn_processors": handler_config["infer_processors"],
        #"label": labels_columns,
        #"feature": features_columns,
        # ⚠️ 关键，给模型提供默认列映射
        #"use_cols": {
        #    "feature": features_columns,
        #   "label": labels_columns
        #}
    },
}


handler_cfg = dataset_task['kwargs']['handler']

# 修正时间段 和自定义数据时间段一致 否则会用默认路径时间 重要！！！
handler_cfg['kwargs'].update({
    'start_time': '2025-01-02 09:30:00',
    'end_time': '2025-09-29 14:59:00',
    'fit_start_time': '2025-01-02 09:30:00',
    'fit_end_time': '2025-04-30 14:59:00'
})

dataset_task['kwargs']['handler'] = handler_cfg

# 模型 task
model_task = {
    "class": "LGBModel",
    "module_path": "qlib.contrib.model.gbdt",
    "kwargs": {
        "loss": "mse",
        "learning_rate": 0.05,
        "n_estimators": 200,
        "num_leaves": 63,
    },
}

# 整体 task
task = {
    "dataset": dataset_task,
    "model": model_task,
}


In [14]:
print(dataset_task)

{'class': 'DatasetH', 'module_path': 'qlib.data.dataset', 'kwargs': {'handler': {'class': 'HighFreqHandler', 'module_path': 'highfreq_handler', 'kwargs': {'start_time': '2025-01-02 09:30:00', 'end_time': '2025-09-29 14:59:00', 'fit_start_time': '2025-01-02 09:30:00', 'fit_end_time': '2025-04-30 14:59:00', 'instruments': ['SH000300', 'SH600105', 'SH600126', 'SH600246', 'SH600353', 'SH600397', 'SH600410', 'SH600530', 'SH600580', 'SH600589', 'SH600590', 'SH600592', 'SH600610', 'SH600698', 'SH600735', 'SH600967', 'SH601086', 'SH601606', 'SH601869', 'SH603011', 'SH603040', 'SH603063', 'SH603072', 'SH603083', 'SH603086', 'SH603090', 'SH603119', 'SH603124', 'SH603127', 'SH603130', 'SH603166', 'SH603171', 'SH603200', 'SH603226', 'SH603228', 'SH603256', 'SH603257', 'SH603286', 'SH603300', 'SH603308', 'SH603316', 'SH603319', 'SH603359', 'SH603382', 'SH603516', 'SH603579', 'SH603630', 'SH603657', 'SH603667', 'SH603683', 'SH603686', 'SH603716', 'SH603758', 'SH603767', 'SH603800', 'SH603809', 'SH60

zerosize报错是因为dataset_task的时间段定义没有修改好  下次哪里报错打印哪里

实例化数据集（这时候 dataset_task 已经定义）

In [9]:
from qlib.utils import init_instance_by_config

dataset = init_instance_by_config(dataset_task)
xtrain, ytrain = dataset.prepare("train", n_jobs=1)
xtest, ytest = dataset.prepare("test", n_jobs=1)

if xtrain.size == 0 or xtest.size == 0:
    raise ValueError("❌ 数据为空！请检查股票池与 provider_uri 是否匹配天勤数据")
else:
    print(f"✅ 数据检查通过: 训练集 {xtrain.shape}, 测试集 {xtest.shape}")

print("训练集:", xtrain.shape, ytrain.shape)
print("样本预览:")
print(xtrain.head())

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import os
import copy
from qlib.utils import init_instance_by_config

# 分批大小
batch_size = 50

# 输出目录
output_dir = "C:/Users/ASUS/qlib_data/highfreq_batches"
os.makedirs(output_dir, exist_ok=True)

# 直接使用已有股票池 stock_codes 或 filtered_stock_codes
stocks = stock_codes  # 或者 filtered_stock_codes

# 遍历分批处理
for i in range(0, len(stocks), batch_size):
    batch_codes = stocks[i:i+batch_size]
    print(f"🔹 处理股票批次 {i//batch_size + 1}: {len(batch_codes)} 支股票")

    # 临时深拷贝 dataset_task，不修改原对象
    tmp_dataset_task = copy.deepcopy(dataset_task)
    tmp_dataset_task["kwargs"]["handler"]["kwargs"]["instruments"] = batch_codes

    # 实例化 Dataset
    dataset = init_instance_by_config(tmp_dataset_task)

    # 准备训练和测试数据，单线程
    xtrain, ytrain = dataset.prepare("train", n_jobs=1)
    xtest, ytest = dataset.prepare("test", n_jobs=1)

    if xtrain.size == 0 or xtest.size == 0:
        print(f"⚠️ 批次 {i//batch_size + 1} 数据为空！跳过")
        continue

    # 保存 Parquet
    xtrain.to_parquet(os.path.join(output_dir, f"xtrain_batch{i//batch_size + 1}.parquet"))
    ytrain.to_parquet(os.path.join(output_dir, f"ytrain_batch{i//batch_size + 1}.parquet"))
    xtest.to_parquet(os.path.join(output_dir, f"xtest_batch{i//batch_size + 1}.parquet"))
    ytest.to_parquet(os.path.join(output_dir, f"ytest_batch{i//batch_size + 1}.parquet"))

    print(f"✅ 批次 {i//batch_size + 1} 已保存")

    # 释放内存
    del dataset, xtrain, ytrain, xtest, ytest


🔹 处理股票批次 1: 50 支股票


Exception in thread Thread-17 (_handle_tasks):
Traceback (most recent call last):
  File "c:\Users\ASUS\miniconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\ASUS\miniconda3\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\ASUS\miniconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ASUS\miniconda3\Lib\multiprocessing\pool.py", line 562, in _handle_tasks
    outqueue.put(None)
    ~~~~~~~~~~~~^^^^^^
  File "c:\Users\ASUS\miniconda3\Lib\site-packages\joblib\pool.py", line 155, in send
    self._writer.send_bytes(buffer.getvalue())
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ASUS\miniconda3\Lib\multiprocessing\connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
    ~~~~~~~~~~~~~~~~^^^^^

训练

In [ ]:

from qlib.utils import init_instance_by_config

#dataset = init_instance_by_config(task["dataset"])
model = init_instance_by_config(task["model"])
recorder = R.get_recorder()
# 开始训练
model.fit(dataset)

# 保存模型
R.save_objects(model=model)

print("✅ 模型训练完成")


[12344:MainThread](2025-10-12 11:30:48,374) INFO - qlib.timer - [log.py:127] - Time cost: 109.438s | Loading data Done
[12344:MainThread](2025-10-12 11:31:28,364) INFO - qlib.timer - [log.py:127] - Time cost: 37.799s | HighFreqNorm Done
[12344:MainThread](2025-10-12 11:31:28,953) INFO - qlib.timer - [log.py:127] - Time cost: 40.578s | fit & process data Done
[12344:MainThread](2025-10-12 11:31:28,954) INFO - qlib.timer - [log.py:127] - Time cost: 150.018s | Init data Done
c:\Users\ASUS\miniconda3\Lib\site-packages\lightgbm\callback.py:347: UserWarning: Only training set found, disabling early stopping.
  _log_warning("Only training set found, disabling early stopping.")


[20]	train's l2: 3.28151e-06
[40]	train's l2: 3.25302e-06
[60]	train's l2: 3.23415e-06
[80]	train's l2: 3.22036e-06
[100]	train's l2: 3.21197e-06
[120]	train's l2: 3.20643e-06
[140]	train's l2: 3.20139e-06
[160]	train's l2: 3.19765e-06
[180]	train's l2: 3.19442e-06
[200]	train's l2: 3.1913e-06


[12344:MainThread](2025-10-12 11:33:11,061) INFO - qlib.workflow - [exp.py:258] - Experiment 163818717121484802 starts running ...
[12344:MainThread](2025-10-12 11:33:11,220) INFO - qlib.workflow - [recorder.py:345] - Recorder eff667654805499d87135ce2286108ad starts running under Experiment 163818717121484802 ...


✅ 模型训练完成


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

KQ_DATA_DIR = Path(r"C:\Users\ASUS\qlib_data")   # 调整为你的路径
FEATURES_DIR = KQ_DATA_DIR / "features"
CAL_FILE = KQ_DATA_DIR / "calendars" / "1min.txt"
INST_FILE = KQ_DATA_DIR / "instruments" / "all.txt"

print("calendar:", CAL_FILE.exists(), CAL_FILE)
print("features dir:", FEATURES_DIR.exists(), FEATURES_DIR)
print("instruments file:", INST_FILE.exists(), INST_FILE)
print()

# 读取 calendar 长度
cal_lines = []
if CAL_FILE.exists():
    with open(CAL_FILE, "r", encoding="utf-8") as f:
        cal_lines = [ln.strip() for ln in f if ln.strip()]
print("calendar timestamps:", len(cal_lines))

# 读取 instruments（简单解析）
insts = []
if INST_FILE.exists():
    try:
        df_inst = pd.read_csv(INST_FILE, header=None)
        # 如果是三列 TSV 或 CSV： instrument, start, end
        if df_inst.shape[1] >= 3:
            insts = df_inst.iloc[:,0].astype(str).tolist()
        else:
            insts = df_inst.iloc[:,0].astype(str).tolist()
    except Exception:
        insts = [ln.split(",")[0].strip() for ln in INST_FILE.read_text(encoding="utf-8").splitlines() if ln.strip()]
print("instruments count:", len(insts))
print()

# 快速检查一批标的（前 200 / 全部可能很慢）
to_check = insts[:200] if len(insts) > 200 else insts

bad = []
summary = []
for inst in to_check:
    # normalize folder name if needed, accept SSE.XXX or sse.xxx or 'SSE_603072' etc.
    inst_dirname = inst.replace(".", "_").replace("/", "_")
    # try common dir naming: sse.xxxx -> sse_xxxx or lower-case
    cand_names = [
        inst_dirname.lower(),
        inst_dirname.upper(),
        inst.replace(".", "").lower(),
        inst.replace(".", "").upper(),
        inst.lower().replace("sse_", "sse.").replace("szse_", "szse.")
    ]
    # find an actual existing folder inside FEATURES_DIR
    found = None
    for p in FEATURES_DIR.iterdir():
        name = p.name.lower()
        if any(name == c for c in cand_names):
            found = p
            break
    if found is None:
        # fallback: try matching ignoring prefix
        base = inst.split(".")[-1]
        for p in FEATURES_DIR.iterdir():
            if base in p.name:
                found = p
                break

    if found is None or not found.is_dir():
        bad.append((inst, "NO_FOLDER"))
        continue

    files = list(found.glob("*.1min.bin"))
    names = [f.name for f in files]
    lengths = {}
    for f in files:
        try:
            arr = np.fromfile(f, dtype=np.float32)
            lengths[f.name] = arr.size
        except Exception as e:
            lengths[f.name] = f"ERR:{e}"
    # record
    summary.append((inst, found.name, len(files), lengths))
    # check problems
    lens = [v for v in lengths.values() if isinstance(v, int)]
    if len(lens) == 0:
        bad.append((inst, "NO_BIN"))
    else:
        # if any length is zero or mismatch
        if any(l <= 0 for l in lens):
            bad.append((inst, "ZERO_LEN", lengths))
        if len(set(lens)) > 1:
            bad.append((inst, "LEN_MISMATCH", lengths))

print("checked", len(to_check))
print("bad examples:", bad[:10])
print("summary sample (first 10):")
for s in summary[:10]:
    print(s)


calendar: True C:\Users\ASUS\qlib_data\calendars\1min.txt
features dir: True C:\Users\ASUS\qlib_data\features
instruments file: True C:\Users\ASUS\qlib_data\instruments\all.txt

calendar timestamps: 43680
instruments count: 501

checked 200
bad examples: [('SH000300\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600105\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600126\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600246\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600353\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600397\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600410\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600530\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600580\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER'), ('SH600589\t2025-01-02 09:30:00\t2025-09-29 14:59:00', 'NO_FOLDER')]
summary sample (first 10):
